In [5]:
from google.colab import drive
# Tenta desmontar o Drive para garantir uma nova conexão limpa
try:
    drive.flush_and_unmount()
except:
    pass
# Monta o Drive novamente. Você deve ver um link e pedir a permissão.
drive.mount('/content/drive')

Mounted at /content/drive


# **Média aritmética de hora em hora**

# **Outra opçao a ser considerado sem o cabeçalho**

In [10]:
import pandas as pd
import os
from google.colab import drive
import numpy as np
import re
from typing import Optional

# ====================================================================
# 1. Montagem, Caminhos e Limpeza
# ====================================================================

print("Montando Google Drive...")
# O código de montagem do drive foi mantido para fins de execução em ambiente Colab
if not os.path.exists('/content/drive'):
    # drive.mount('/content/drive') # Descomente se precisar montar o drive aqui
    pass
else:
    print("Drive já montado.")

PASTA_DRIVE = '/content/drive/MyDrive/Colab Notebooks/Projeto_dissertacao'

# --- Solicita o nome do arquivo de entrada ---
NOME_ARQUIVO_ENTRADA = input("Digite o nome do arquivo TXT ou CSV para análise (Ex: PicoW.txt, S1.csv): ").strip()

if '.' not in NOME_ARQUIVO_ENTRADA:
    NOME_ARQUIVO_ENTRADA += '.txt'

ARQUIVO_ENTRADA = NOME_ARQUIVO_ENTRADA
CAMINHO_SAVE_ESTRUTURADO = os.path.join(PASTA_DRIVE, 'dados_estruturados.csv')
CAMINHO_COMPLETO_ENTRADA = os.path.join(PASTA_DRIVE, ARQUIVO_ENTRADA)

NOME_LOG = ARQUIVO_ENTRADA.rsplit('.', 1)[0].upper()

print(f"Caminho do arquivo de entrada definido: {CAMINHO_COMPLETO_ENTRADA}")
print(f"Caminho de saída estruturado definido: {CAMINHO_SAVE_ESTRUTURADO}")

# --- LIMPEZA DE RESULTADOS ANTERIORES ---
if os.path.exists(CAMINHO_SAVE_ESTRUTURADO):
    try:
        os.remove(CAMINHO_SAVE_ESTRUTURADO)
        print(f"✅ Arquivo de resultados anteriores removido: {os.path.basename(CAMINHO_SAVE_ESTRUTURADO)}")
    except Exception as e:
        print(f"AVISO: Não foi possível remover o arquivo de resultados anteriores. {e}")
print("-" * 50)


# ====================================================================
# 2. FUNÇÕES DE ESTRUTURAÇÃO DE DADOS
# ====================================================================

def processar_dataframe(df: pd.DataFrame, nome_log: str, caminho_saida: str) -> bool:
    """Função de processamento final e salvamento do DataFrame estruturado."""
    if 'Dispositivo' not in df.columns:
        df['Dispositivo'] = nome_log

    # Limpeza e garantia de tipos
    # A coluna 'Timestamp' é agora a 'Horario' extraída do log (ou 'Timestamp' do CSV)
    if 'Timestamp' not in df.columns:
        if 'Horario' in df.columns:
             df.rename(columns={'Horario': 'Timestamp'}, inplace=True)
        else:
             print("ERRO: Coluna 'Timestamp' ou 'Horario' não encontrada após estruturação.")
             return False

    df.dropna(subset=['Timestamp'], inplace=True)
    if df.empty: return False

    df['Dispositivo'].replace('', np.nan, inplace=True)
    df.dropna(subset=['Dispositivo'], inplace=True)
    if df.empty: return False

    # pd.to_datetime é movido para o processamento específico (regex/csv)
    df.set_index('Timestamp', inplace=True)
    df['Temperatura'] = pd.to_numeric(df['Temperatura'], errors='coerce')
    df['Umidade'] = pd.to_numeric(df['Umidade'], errors='coerce')
    df.dropna(subset=['Temperatura', 'Umidade'], inplace=True)

    if df.empty: return False

    # Salvamento no arquivo estruturado (append ou novo)
    colunas_salvamento = ['Dispositivo', 'Temperatura', 'Umidade']
    if os.path.exists(caminho_saida):
        df[colunas_salvamento].to_csv(caminho_saida, mode='a', header=False)
    else:
        df[colunas_salvamento].to_csv(caminho_saida)

    print(f"Estruturação de {nome_log} concluída. Salvo em: {os.path.basename(caminho_saida)} (Total de {len(df)} linhas)")
    return True

def estruturar_log_regex(caminho_entrada: str, caminho_saida: str, nome_log: str) -> bool:
    """CORREÇÃO: Tenta estruturar o log usando expressões regulares para o formato Pico W."""
    print(f"\nTentando estruturação (Expressão Regular - Log Pico W) de: {os.path.basename(caminho_entrada)}")

    # O padrão de expressão regular para extrair os dados do seu log
    # Ele captura: Log_Timestamp, Dispositivo, Leitura, Horario, Temp, Umid
    # Note que usamos 'Horario' como a coluna de timestamp principal (o horário da leitura do sensor)
    PADRAO_LOG = re.compile(
        r"\[(?P<Log_Timestamp>.*?)\]\s+"
        r"Dispositivo: (?P<Dispositivo>.*?), "
        r"Leitura: (?P<Leitura>\d+), "
        r"Horario: (?P<Horario>.*?), "
        r"Temp: (?P<Temperatura>[\d\.]+), "
        r"Umid: (?P<Umidade>[\d\.]+)"
    )

    dados = []

    try:
        with open(caminho_entrada, 'r', encoding='utf-8', errors='ignore') as f:
            for linha in f:
                match = PADRAO_LOG.search(linha)
                if match:
                    dados.append(match.groupdict())
    except Exception as e:
        print(f"ERRO durante a leitura do arquivo TXT com regex: {e}")
        return False

    if dados:
        df_estruturado = pd.DataFrame(dados)

        # Converte as colunas de tempo e números
        # Usamos 'Horario' como o Timestamp principal para fins de Resample.
        df_estruturado['Timestamp'] = pd.to_datetime(df_estruturado['Horario'], errors='coerce')
        df_estruturado['Temperatura'] = pd.to_numeric(df_estruturado['Temperatura'], errors='coerce')
        df_estruturado['Umidade'] = pd.to_numeric(df_estruturado['Umidade'], errors='coerce')
        df_estruturado.dropna(subset=['Timestamp', 'Temperatura', 'Umidade'], inplace=True)

        if not df_estruturado.empty:
            # Seleciona apenas as colunas necessárias para o processamento final
            df_processamento = df_estruturado[['Dispositivo', 'Timestamp', 'Temperatura', 'Umidade']].copy()
            return processar_dataframe(df_processamento, nome_log, caminho_saida)

    print(f"AVISO REGEX: Nenhuma linha de dados pôde ser extraída com o padrão de log. Tentando CSV/Tabular...")
    return False

def estruturar_log_csv(caminho_entrada: str, caminho_saida: str, nome_log: str) -> bool:
    """Tenta estruturar o log usando formatos CSV ou tabulares comuns."""
    print(f"\nTentando estruturação (CSV/Tabular) de: {os.path.basename(caminho_entrada)}")
    df: Optional[pd.DataFrame] = None

    # Tentativa 1: Arquivo sem cabeçalho (Formato: Contagem, Data_Hora, Temperatura, Umidade)
    COLUNAS_BRUTAS_IMAGEM1 = ['Contagem_Col', 'Data_Hora_Col', 'Temperatura_Col', 'Umidade_Col']
    # Tentativa com ponto e vírgula primeiro, já que é comum em logs PT-BR
    separadores = [(';', 'CSV (;)'), (',', 'CSV (,)')]

    for sep_char, sep_nome in separadores:
        try:
            df_bruto = pd.read_csv(caminho_entrada, sep=sep_char, header=None,
                                     names=COLUNAS_BRUTAS_IMAGEM1,
                                     encoding='latin-1', engine='python', on_bad_lines='skip',
                                     skiprows=0)

            if df_bruto.shape[1] >= 4 and df_bruto.shape[0] > 0:
                df = df_bruto.iloc[:, :4].copy()
                df.rename(columns={'Data_Hora_Col': 'Timestamp', 'Temperatura_Col': 'Temperatura', 'Umidade_Col': 'Umidade'}, inplace=True)

                # Tratamento de timestamp (removendo vírgulas e espaços extras)
                df['Timestamp'] = df['Timestamp'].astype(str).str.replace(',', ' ', regex=False).str.strip()
                df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')

                df['Dispositivo'] = nome_log
                df = df[['Dispositivo', 'Timestamp', 'Temperatura', 'Umidade']].copy()

                if not df.dropna(subset=['Timestamp']).empty:
                    return processar_dataframe(df, nome_log, caminho_saida)

        except Exception:
             continue # Tenta o próximo separador ou formato

    # Tentativa 2: Arquivo com cabeçalho (Procura por colunas específicas)
    for sep_char, _ in separadores:
        try:
            df_teste = pd.read_csv(caminho_entrada, sep=sep_char,
                                     engine='python', on_bad_lines='skip',
                                     encoding='latin-1')

            colunas_presentes = set(df_teste.columns)
            temp_match = 'Temperatura' in colunas_presentes or 'Temperatura(C)' in colunas_presentes
            umid_match = 'Umidade_%' in colunas_presentes or 'Umidade(%)' in colunas_presentes

            if ('Data_Hora_Coleta' in colunas_presentes or 'Timestamp' in colunas_presentes) and temp_match and umid_match:
                df = df_teste.copy()
                df.rename(columns={'Data_Hora_Coleta': 'Timestamp',
                                       'Temperatura(C)': 'Temperatura',
                                       'Umidade_%': 'Umidade',
                                       'Umidade(%)': 'Umidade'}, inplace=True)

                # Tenta formatar a data
                if df['Timestamp'].dtype == object:
                      df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%d/%m/%Y %H:%M', errors='coerce')

                if 'Dispositivo' not in df.columns:
                    df['Dispositivo'] = nome_log

                df = df[['Dispositivo', 'Timestamp', 'Temperatura', 'Umidade']].copy()
                return processar_dataframe(df, nome_log, caminho_saida)

        except Exception:
             continue

    print(f"AVISO TABULAR/CSV: O arquivo não pôde ser lido corretamente com os padrões conhecidos.")
    return False

# --- EXECUÇÃO DA ESTRUTURAÇÃO (TENTATIVA MÚLTIPLA) ---

# Tenta a Expressão Regular (nova função) primeiro, pois é o formato de log conhecido
if not estruturar_log_regex(CAMINHO_COMPLETO_ENTRADA, CAMINHO_SAVE_ESTRUTURADO, NOME_LOG):
    # Se falhar, tenta formatos CSV/Tabulares
    estruturar_log_csv(CAMINHO_COMPLETO_ENTRADA, CAMINHO_SAVE_ESTRUTURADO, NOME_LOG)


# ====================================================================
# 3. Carregamento Final
# ====================================================================

df_dados = pd.DataFrame()

try:
    df_dados = pd.read_csv(CAMINHO_SAVE_ESTRUTURADO, index_col=0, parse_dates=True)
    df_dados.rename(columns={'Temperatura': 'Temp', 'Umidade': 'Umid'}, inplace=True)
    df_dados['Temp'] = pd.to_numeric(df_dados['Temp'], errors='coerce')
    df_dados['Umid'] = pd.to_numeric(df_dados['Umid'], errors='coerce')
    df_dados.dropna(subset=['Temp', 'Umid'], inplace=True)

except FileNotFoundError:
    print("\nERRO FATAL: O arquivo estruturado não foi encontrado. A estruturação falhou.")
    raise SystemExit(0)
except Exception as e:
    print(f"\nERRO: Falha ao carregar o arquivo estruturado. {e}")
    raise SystemExit(0)

if df_dados.empty:
    print(f"\nERRO FATAL: O DataFrame {NOME_LOG} está vazio. Verifique o formato do arquivo de entrada e os separadores usados.")
    raise SystemExit(0)
else:
    print(f"\nDataFrame {NOME_LOG} carregado com sucesso. Iniciando Análise.")

# ====================================================================
# 4. Cálculo das Médias ARITMÉTICAS HORÁRIAS (Resample com Contagem)
# ====================================================================

df_medias_horarias_todos = pd.DataFrame()
resultados_analise = []
grupos_dispositivo = df_dados.groupby('Dispositivo')

print("\n--- ⏳ Calculando Médias ARITMÉTICAS HORÁRIAS (Redondas) ---")

for dispositivo, df_grupo in grupos_dispositivo:
    if len(df_grupo) < 1:
        continue

    # 1. RESAMPLE: Agrupa os dados por hora ('H') e calcula a média e a contagem.
    # Usamos agg para calcular a média e a contagem simultaneamente.
    df_medias_horarias_agg = df_grupo[['Temp', 'Umid']].resample('H').agg(['mean', 'count']).dropna()

    if df_medias_horarias_agg.empty:
        continue

    # Renomear e estruturar o resultado para o formato desejado
    df_medias_horarias = pd.DataFrame({
        'Temp_Média_H': df_medias_horarias_agg['Temp']['mean'],
        'Umid_Média_H': df_medias_horarias_agg['Umid']['mean'],
        'Contagem': df_medias_horarias_agg['Temp']['count'] # Assume que a contagem de Temp é a contagem de pontos
    })

    # 2. MÉDIA FINAL: Calcula a média aritmética dessas médias horárias.
    media_temp_final = df_medias_horarias['Temp_Média_H'].mean()
    media_umid_final = df_medias_horarias['Umid_Média_H'].mean()

    # 3. CONSOLIDAÇÃO: Armazena as médias horárias e a contagem para o cálculo da média geral
    df_medias_horarias['Dispositivo'] = dispositivo
    df_medias_horarias_todos = pd.concat([df_medias_horarias_todos, df_medias_horarias])

    resultados_analise.append({
        'Dispositivo': dispositivo,
        'Temp_Média_Geral': media_temp_final,
        'Umid_Média_Geral': media_umid_final,
        'Total_Horas_Analisadas': len(df_medias_horarias)
    })

df_resultados = pd.DataFrame(resultados_analise).set_index('Dispositivo')
df_resultados.dropna(inplace=True)


# ====================================================================
# 5. Apresentação dos Resultados (Com Contagem)
# ====================================================================

# 1. Apresentação da Tabela Hora a Hora
if not df_medias_horarias_todos.empty:

    # ------------------------------------------------------------------------------------
    # FORMATO DA IMAGEM 1: Média Hora a Hora (Com a Contagem de Pontos de Dados)
    # ------------------------------------------------------------------------------------
    print("\n" + "="*90)
    print(f"|   ANÁLISE HORA A HORA do arquivo: {ARQUIVO_ENTRADA}                                          |")
    print("="*90)

    print("\n--- 📊 Médias ARITMÉTICAS por Hora Redonda (com Contagem de Pontos) ---")

    df_display_horario = df_medias_horarias_todos.reset_index()
    df_display_horario['Hora_Redonda'] = df_display_horario['Timestamp'].dt.strftime('%d/%m %H:00')

    # Colunas de exibição: Dispositivo, Hora, Média Temp, Média Umid, Contagem
    tabela_display = df_display_horario[['Dispositivo', 'Hora_Redonda', 'Temp_Média_H', 'Umid_Média_H', 'Contagem']]

    # Imprime a tabela com formatação Markdown
    print(tabela_display.to_markdown(index=False, floatfmt=".2f"))

    print("-" * 50)

    # ------------------------------------------------------------------------------------
    # FORMATO DA IMAGEM 2: Média Final (Média das Médias Horárias)
    # ------------------------------------------------------------------------------------

    # Cálculo da Média Aritmética GERAL de TODAS as médias horárias
    media_temp_geral_final = df_medias_horarias_todos['Temp_Média_H'].mean()
    media_umid_geral_final = df_medias_horarias_todos['Umid_Média_H'].mean()

    # Média das Médias Horárias por Dispositivo
    print("\n--- 🌟 Média ARITMÉTICA FINAL (Média das Médias Horárias) por Dispositivo ---")

    # Renomeia colunas para manter a estética do print original
    df_resultados.rename(columns={
        'Temp_Média_Geral': 'Temp_Média_Aritmetica',
        'Umid_Média_Geral': 'Umid_Média_Aritmetica',
        'Total_Horas_Analisadas': 'Total_Horas'
    }, inplace=True)

    print(df_resultados[['Temp_Média_Aritmetica', 'Umid_Média_Aritmetica', 'Total_Horas']].to_markdown(floatfmt=".2f"))


    # Média Geral do Arquivo (FINAL)
    print("\n--- ☀️ Média ARITMÉTICA GERAL do Arquivo (Final) ---")
    print(f"|   Temperatura Média GERAL: {media_temp_geral_final:.2f}°C")
    print(f"|   Umidade Média GERAL:     {media_umid_geral_final:.2f}%")
    print("------------------------------------------")

else:
    print("\nAVISO: Não foi possível calcular ou exibir resultados. Verifique os dados no arquivo estruturado.")

Montando Google Drive...
Drive já montado.
Digite o nome do arquivo TXT ou CSV para análise (Ex: PicoW.txt, S1.csv): DADOS_13.CSV
Caminho do arquivo de entrada definido: /content/drive/MyDrive/Colab Notebooks/Projeto_dissertacao/DADOS_13.CSV
Caminho de saída estruturado definido: /content/drive/MyDrive/Colab Notebooks/Projeto_dissertacao/dados_estruturados.csv
✅ Arquivo de resultados anteriores removido: dados_estruturados.csv
--------------------------------------------------

Tentando estruturação (Expressão Regular - Log Pico W) de: DADOS_13.CSV
AVISO REGEX: Nenhuma linha de dados pôde ser extraída com o padrão de log. Tentando CSV/Tabular...

Tentando estruturação (CSV/Tabular) de: DADOS_13.CSV
Estruturação de DADOS_13 concluída. Salvo em: dados_estruturados.csv (Total de 1 linhas)

DataFrame DADOS_13 carregado com sucesso. Iniciando Análise.

--- ⏳ Calculando Médias ARITMÉTICAS HORÁRIAS (Redondas) ---

|   ANÁLISE HORA A HORA do arquivo: DADOS_13.CSV                               

/tmp/ipython-input-2875818440.py:159: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
/tmp/ipython-input-2875818440.py:68: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Dispositivo'].replace('', np.nan, inplace=True)
/tmp/ipython-input-2875818440.py:254: FutureWarning: 'H' is deprecated and will be removed in a future ver